In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [5]:
SOURCE_DIR = "."
TRAIN_DIR = "data/train"
VAL_DIR = "data/val"

import os, shutil, random

for category in ["normal", "disease"]: 
    src = os.path.join(SOURCE_DIR, category)
    images = os.listdir(src)
    random.shuffle(images)

    split = int(len(images) * 0.8)

    train_imgs = images[:split]
    val_imgs = images[split:]

    os.makedirs(os.path.join(TRAIN_DIR, category), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, category), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(src, img), os.path.join(TRAIN_DIR, category, img))

    for img in val_imgs:
        shutil.copy(os.path.join(src, img), os.path.join(VAL_DIR, category, img))

print("Dataset split complete")

Dataset split complete


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16

train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "data/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    "data/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
output = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

model.save("skin_model.h5")

print("Model saved ✅")

Found 932 images belonging to 2 classes.
Found 234 images belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 130ms/step - accuracy: 0.9496 - loss: 0.1254 - val_accuracy: 0.9872 - val_loss: 0.0340
Epoch 2/3
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - accuracy: 0.9957 - loss: 0.0160 - val_accuracy: 0.9829 - val_loss: 0.0518
Epoch 3/3
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.9989 - loss: 0.0079 - val_accuracy: 0.9872 - val_loss: 0.0218


Model saved ✅


In [20]:
import tensorflow as tf
import cv2
import numpy as np

model = tf.keras.models.load_model("skin_model.h5")

img = cv2.imread("test2.jpg")  # 👈 put your test image
img = cv2.resize(img, (224,224))
img = img / 255.0
img = np.reshape(img, (1,224,224,3))

pred = model.predict(img)[0][0]

print("Raw value:", pred)

if pred < 0.5:
    print(f"🧴 Disease Detected ({(1-pred)*100:.2f}%)")
else:
    print(f"✅ Normal Skin ({pred*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 634ms/step
Raw value: 0.999982
✅ Normal Skin (100.00%)


In [14]:
import cv2

img = cv2.imread("test.jpg")
print(img is None)

True


In [16]:
print(train_data.class_indices)

{'disease': 0, 'normal': 1}
